In [ ]:
# ============================================
# SECTION 2: FEATURE ENGINEERING
# ============================================

def create_all_features(df):
    """Crée 15 nouvelles features"""
    df_feat = df.copy()
    
    # QualityArea (ML Mastery)
    df_feat['QualityArea'] = df_feat['OverallQual'] * df_feat['GrLivArea']
    
    # Âges
    df_feat['HouseAge'] = df_feat['YrSold'] - df_feat['YearBuilt']
    df_feat['RemodAge'] = df_feat['YrSold'] - df_feat['YearRemodAdd']
    
    # Surfaces totales
    df_feat['TotalSF'] = df_feat['TotalBsmtSF'] + df_feat['1stFlrSF'] + df_feat['2ndFlrSF']
    df_feat['TotalPorchSF'] = (df_feat['OpenPorchSF'] + df_feat['EnclosedPorch'] + 
                                df_feat['3SsnPorch'] + df_feat['ScreenPorch'])
    df_feat['TotalPropertySF'] = df_feat['LotArea'] + df_feat['TotalSF']
    
    # Salles de bain
    df_feat['TotalBath'] = (df_feat['FullBath'] + 0.5*df_feat['HalfBath'] +
                            df_feat['BsmtFullBath'] + 0.5*df_feat['BsmtHalfBath'])
    
    # Ratio
    df_feat['BedroomBathRatio'] = df_feat['BedroomAbvGr'] / (df_feat['TotalBath'] + 0.01)
    
    # Qualité combinée
    df_feat['QualityCond'] = df_feat['OverallQual'] * df_feat['OverallCond']
    
    # Log transformation
    df_feat['LogLotArea'] = np.log1p(df_feat['LotArea']) #skewness 12.5 before log and 0.5 after 
    
    # Variables binaires
    df_feat['HasBasement'] = (df_feat['TotalBsmtSF'] > 0).astype(int)
    df_feat['HasGarage'] = (df_feat['GarageArea'] > 0).astype(int)
    df_feat['HasFireplace'] = (df_feat['Fireplaces'] > 0).astype(int)
    df_feat['HasPool'] = (df_feat['PoolArea'] > 0).astype(int)
    
    return df_feat

print(" Fonction create_all_features() définie")

In [ ]:
data_feat = create_all_features(data)

In [ ]:
# ============================================
# SPLIT TRAIN/TEST (SUR LE TRAIN KAGGLE)
# ============================================

from sklearn.model_selection import train_test_split

# 1. Séparer X (features) et y (cible)
X = data_feat.drop('SalePrice', axis=1)
y = data_feat['SalePrice']

# 2. Split 80% / 20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,          # 20% pour le test
    random_state=42
)

print(f"Train: {X_train.shape[0]} lignes")
print(f"Test: {X_test.shape[0]} lignes")

In [ ]:
# ============================================
# PREPROCESSING SIMPLIFIÉ (SANS PIPELINE)
# ============================================

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np

print("="*60)
print("PREPROCESSING SIMPLIFIÉ")
print("="*60)

# ============================================
# 1. DÉFINITION DES COLONNES
# ============================================

# Colonnes où NaN signifie "absence" → on impute par "None"
none_cols = [
    "PoolQC", "MiscFeature", "Fence", "FireplaceQu", "Alley",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]

# Colonnes avec une seule valeur manquante → imputer par le mode
mode_cols = ["Electrical"]

# Toutes les colonnes catégorielles
cat_cols = none_cols + mode_cols

# Colonnes numériques
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'Id' in num_cols:
    num_cols.remove('Id')

print(f"🔹 Colonnes catégorielles : {len(cat_cols)}")
print(f"🔹 Colonnes numériques : {len(num_cols)}")

# ============================================
# 2. ONEHOTENCODER (ignore les valeurs inconnues)
# ============================================

# 2a. Imputer les catégories avec 'missing'
for col in cat_cols:
    X_train[col] = X_train[col].fillna('missing').astype(str)
    X_test[col] = X_test[col].fillna('missing').astype(str)

# 2b. OneHotEncoder
onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 2c. Appliquer sur les colonnes catégorielles
X_train_cat = onehot.fit_transform(X_train[cat_cols])
X_test_cat = onehot.transform(X_test[cat_cols])

print(f"✅ Catégories encodées : {X_train_cat.shape[1]} colonnes")

# ============================================
# 3. IMPUTATION ET STANDARDISATION DES NUMÉRIQUES
# ============================================

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_num = scaler.fit_transform(imputer.fit_transform(X_train[num_cols]))
X_test_num = scaler.transform(imputer.transform(X_test[num_cols]))

print(f"✅ Numériques standardisées : {X_train_num.shape[1]} colonnes")

# ============================================
# 4. RÉASSEMBLER
# ============================================

X_train_processed = np.hstack([X_train_num, X_train_cat])
X_test_processed = np.hstack([X_test_num, X_test_cat])

print(f"\n✅ Train après preprocessing : {X_train_processed.shape}")
print(f"✅ Test après preprocessing : {X_test_processed.shape}")

In [ ]:
# ============================================
# SOLUTION FINALE : MAX_FEATURES
# ============================================

from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso

# Choisir 50 features (ajustable)
N_FEATURES = 60

lasso = Lasso(alpha=0.001, max_iter=10000, random_state=42)
selector = SelectFromModel(lasso, max_features=N_FEATURES)
selector.fit(X_train_processed, y_train)

selected_mask = selector.get_support()
X_train_selected = X_train_processed[:, selected_mask]
X_test_selected = X_test_processed[:, selected_mask]

print(f"✅ Features avant sélection : {X_train_processed.shape[1]}")
print(f"✅ Features après sélection : {X_train_selected.shape[1]}")

# ============================================
# SAUVEGARDE DES DONNÉES PRÉPARÉES
# ============================================

In [ ]:
import joblib
import numpy as np

# 1. Sauvegarder les données sélectionnées
# À la fin du notebook de preprocessing
np.save('../data/X_train_processed.npy', X_train_processed)
np.save('../data/X_test_processed.npy', X_test_processed)
np.save('../data/X_train_selected.npy', X_train_selected)
np.save('../data/X_test_selected.npy', X_test_selected)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)

# 2. Sauvegarder aussi les objets de preprocessing (pour le test Kaggle)
joblib.dump(onehot, '../data/onehot_encoder.joblib')
joblib.dump(imputer, '../data/imputer.joblib')
joblib.dump(scaler, '../data/scaler.joblib')
joblib.dump(selector, '../data/selector.joblib')

